# BirdCLEF+ 2026 ONNX Perch Speed Test

Runtime-only notebook for testing whether ONNX Perch can replace the slower TensorFlow Perch scoring path. This notebook does not train, blend, calibrate, or write a leaderboard submission. Its job is to load ONNX Perch, score soundscape-shaped audio, and report seconds per file.

## 1. Setup And Configuration

Use fixed Kaggle input roots and attached wheels so the notebook remains offline-safe during competition reruns.

In [ ]:
from __future__ import annotations

from pathlib import Path
import subprocess
import sys
import time
from types import ModuleType
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


class CFG:
    """Configuration for ONNX Perch runtime measurement."""

    seed = 42
    data_root = Path("/kaggle/input/competitions/birdclef-2026")
    onnx_root = Path(
        "/kaggle/input/datasets/tuckerarrants/perch-v2-no-dft-onnx"
    )
    output_path = Path("/kaggle/working/onnx_perch_timing.csv")

    onnx_model_patterns = ("perch_v2_no_dft*.onnx", "perch_v2*.onnx")
    onnx_wheel_pattern = "onnxruntime-*.whl"
    sample_rate = 32_000
    clip_seconds = 5
    soundscape_seconds = 60
    n_windows = soundscape_seconds // clip_seconds
    batch_files = 4
    onnx_threads = 4
    public_dry_run_files = 20

    samples_per_window = sample_rate * clip_seconds
    samples_per_file = sample_rate * soundscape_seconds


np.random.seed(CFG.seed)
print("Competition root:", CFG.data_root)
print("ONNX root:", CFG.onnx_root)


In [ ]:
def install_onnxruntime_if_needed() -> ModuleType:
    """Import ONNX Runtime, installing an attached wheel if needed.

    Returns:
        ModuleType: Imported `onnxruntime` module.

    Raises:
        FileNotFoundError: If ONNX Runtime is unavailable and no wheel exists
            under `/kaggle/input`.
    """
    try:
        import onnxruntime as ort

        return ort
    except ImportError:
        wheels = sorted(CFG.onnx_root.glob(CFG.onnx_wheel_pattern))
        if not wheels:
            wheels = sorted(Path("/kaggle/input").glob("**/onnxruntime-*.whl"))
        if not wheels:
            raise FileNotFoundError(
                "No offline onnxruntime wheel found under /kaggle/input."
            )

        wheel = wheels[0]
        print("Installing ONNX Runtime from", wheel)
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-deps",
                str(wheel),
            ],
            check=True,
        )

        import onnxruntime as ort

        return ort


ort = install_onnxruntime_if_needed()
import librosa
import soundfile as sf

print("ONNX Runtime:", ort.__version__)


## 2. Locate Audio And ONNX Perch

Hidden submissions mount `test_soundscapes`. Public dry runs usually do not, so this notebook falls back to a small sample of `train_soundscapes` for timing only.

In [ ]:
sample_submission = pd.read_csv(CFG.data_root / "sample_submission.csv")
target_columns = [c for c in sample_submission.columns if c != "row_id"]

print("Sample submission:", sample_submission.shape)
print("Target columns:", len(target_columns))


def find_onnx_model() -> Path:
    """Find the ONNX Perch model file from attached Kaggle inputs.

    Returns:
        Path: ONNX Perch model path.

    Raises:
        FileNotFoundError: If no matching ONNX model is attached.
    """
    for pattern in CFG.onnx_model_patterns:
        candidates = sorted(CFG.onnx_root.glob(pattern))
        if not candidates:
            candidates = sorted(Path("/kaggle/input").glob(f"**/{pattern}"))
        if candidates:
            return candidates[0]
    raise FileNotFoundError("No ONNX Perch model found under /kaggle/input.")


def list_soundscape_files() -> tuple[list[Path], str]:
    """List hidden test files or public dry-run train files.

    Returns:
        tuple[list[Path], str]: Audio paths and a source label.

    Raises:
        FileNotFoundError: If neither test nor train soundscapes are mounted.
    """
    test_dir = CFG.data_root / "test_soundscapes"
    test_paths = sorted(test_dir.glob("*.ogg")) if test_dir.exists() else []
    if test_paths:
        return test_paths, "test_soundscapes"

    train_dir = CFG.data_root / "train_soundscapes"
    train_paths = sorted(train_dir.glob("*.ogg")) if train_dir.exists() else []
    if train_paths:
        return train_paths[: CFG.public_dry_run_files], "train_soundscapes"

    raise FileNotFoundError("No soundscape OGG files found for timing.")


onnx_model_path = find_onnx_model()
audio_paths, audio_source = list_soundscape_files()
print("ONNX model:", onnx_model_path)
print("Audio source:", audio_source)
print("Audio files:", len(audio_paths))


In [ ]:
def make_session(path: Path) -> ort.InferenceSession:
    """Create an optimized CPU ONNX Runtime session.

    Args:
        path (Path): ONNX model path.

    Returns:
        ort.InferenceSession: Loaded ONNX Runtime CPU session.
    """
    options = ort.SessionOptions()
    options.intra_op_num_threads = CFG.onnx_threads
    options.inter_op_num_threads = 1
    options.graph_optimization_level = (
        ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    )
    return ort.InferenceSession(
        str(path), sess_options=options, providers=["CPUExecutionProvider"]
    )


session = make_session(onnx_model_path)
input_name = session.get_inputs()[0].name
output_names = [output.name for output in session.get_outputs()]

print("Input:", input_name, session.get_inputs()[0].shape)
print("Outputs:", [(out.name, out.shape) for out in session.get_outputs()])


## 3. Audio And Runtime Helpers

Each file is read once, padded or trimmed to 60 seconds, reshaped into 12 raw 5-second chunks, and passed directly to ONNX Perch.

In [ ]:
def load_audio_60s(audio_path: Path) -> np.ndarray:
    """Load one soundscape as a fixed-length mono waveform.

    Args:
        audio_path (Path): Soundscape OGG path.

    Returns:
        np.ndarray: Float32 waveform with exactly 60 seconds of audio.
    """
    audio, sr = sf.read(str(audio_path), dtype="float32", always_2d=False)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    if sr != CFG.sample_rate:
        audio = librosa.resample(
            audio, orig_sr=sr, target_sr=CFG.sample_rate
        ).astype(np.float32)

    if len(audio) < CFG.samples_per_file:
        audio = np.pad(audio, (0, CFG.samples_per_file - len(audio)))
    else:
        audio = audio[: CFG.samples_per_file]
    return audio.astype(np.float32, copy=False)


def file_to_windows(audio_path: Path) -> np.ndarray:
    """Convert one soundscape file into raw 5-second Perch windows.

    Args:
        audio_path (Path): Soundscape OGG path.

    Returns:
        np.ndarray: Array shaped `(12, 160000)` for ONNX Perch input.
    """
    audio = load_audio_60s(audio_path)
    return audio.reshape(CFG.n_windows, CFG.samples_per_window)


def run_perch(batch: np.ndarray) -> dict[str, np.ndarray]:
    """Run ONNX Perch on a batch of raw waveform windows.

    Args:
        batch (np.ndarray): Raw waveform batch shaped `(n, 160000)`.

    Returns:
        dict[str, np.ndarray]: Output arrays keyed by ONNX output name.
    """
    outputs = session.run(None, {input_name: batch.astype(np.float32)})
    return dict(zip(output_names, outputs))


def batched(paths: list[Path], batch_size: int) -> list[list[Path]]:
    """Split file paths into fixed-size batches.

    Args:
        paths (list[Path]): Soundscape files to score.
        batch_size (int): Number of 60-second files per batch.

    Returns:
        list[list[Path]]: File batches.
    """
    return [
        paths[i : i + batch_size] for i in range(0, len(paths), batch_size)
    ]


## 4. Run Speed Test

The timing table separates audio loading plus ONNX inference at the file-batch level. Use the median seconds per file as the main go/no-go signal before creating a blend notebook.

In [ ]:
def score_files_for_timing(paths: list[Path]) -> pd.DataFrame:
    """Measure ONNX Perch runtime over soundscape files.

    Args:
        paths (list[Path]): Soundscape files used for the timing run.

    Returns:
        pd.DataFrame: Batch-level timing summary.
    """
    rows = []
    total_start = time.perf_counter()

    for batch_idx, batch_paths in enumerate(
        batched(paths, CFG.batch_files), start=1
    ):
        start = time.perf_counter()
        windows = np.concatenate(
            [file_to_windows(path) for path in batch_paths], axis=0
        )
        load_seconds = time.perf_counter() - start

        infer_start = time.perf_counter()
        outputs = run_perch(windows)
        infer_seconds = time.perf_counter() - infer_start
        total_seconds = time.perf_counter() - start

        row = {
            "batch": batch_idx,
            "files": len(batch_paths),
            "windows": len(windows),
            "load_seconds": load_seconds,
            "infer_seconds": infer_seconds,
            "total_seconds": total_seconds,
            "seconds_per_file": total_seconds / len(batch_paths),
            "label_shape": str(outputs.get("label", np.array([])).shape),
            "embedding_shape": str(
                outputs.get("embedding", np.array([])).shape
            ),
        }
        rows.append(row)
        print(
            f"Batch {batch_idx}: {row['files']} files, "
            f"{row['seconds_per_file']:.2f}s/file"
        )

    elapsed = time.perf_counter() - total_start
    print(f"Finished {len(paths)} files in {elapsed:.1f}s")
    return pd.DataFrame(rows)


def estimate_hidden_runtime(timing: pd.DataFrame) -> None:
    """Print a projected hidden-test runtime when row count is available.

    Args:
        timing (pd.DataFrame): Batch-level timing summary.
    """
    median_seconds = timing["seconds_per_file"].median()
    print(f"Median seconds per 60s file: {median_seconds:.2f}")

    hidden_rows = len(sample_submission)
    if hidden_rows <= 3:
        print(
            "Public dry run has no hidden row count; projection unavailable."
        )
        return

    hidden_files = hidden_rows / CFG.n_windows
    projected_minutes = hidden_files * median_seconds / 60
    print(f"Projected hidden files: {hidden_files:.0f}")
    print(f"Projected ONNX Perch runtime: {projected_minutes:.1f} minutes")


timing_df = score_files_for_timing(audio_paths)
timing_df.to_csv(CFG.output_path, index=False)
print("Saved", CFG.output_path)

estimate_hidden_runtime(timing_df)
display(timing_df)
